# 07 -- Walk-forward analysis

Paired script: `analysis/walk_forward.py`. Runs the FULL rolling train/test evaluation
(`run`, not just `generate_windows`) and reports each window's train/test win-rate and
R-expectancy -- the actual walk-forward performance table this notebook's name promises.

**Fixed, 2026-07-21 Codex review finding:** this notebook previously imported only
`generate_windows` and checked four date boundaries; it never ran an evaluation or reported
any train/test performance.

**Scope, stated explicitly (matching `walk_forward.py`'s own module docstring): this is a
descriptive rolling-window stability report, not a walk-forward OPTIMIZATION procedure** --
no parameter is selected on the train window and frozen for the test window.

**Uses clearly-labelled SYNTHETIC trade data.** Real-data run: PENDING.

In [ ]:
import sys
import tempfile
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from analysis.walk_forward import run

In [ ]:
# Same purged-boundary-aware fixture hand-traced in tests/test_walk_forward.py
# (t1/t2/t4/t8 offset +12h from the day-integer boundary grid so none of
# them coincidentally lands exactly on a window edge). The overall analysis
# period is anchored at the earliest ENTRY (not earliest exit -- a 2026-07-22
# Codex review fix), so t0's own entry now legitimately anchors window 0
# itself, rather than being purged.
BASE = pd.Timestamp("2026-01-01", tz="UTC")


def row(trade_id, exit_time, exit_price, profit):
    return {
        "trade_id": trade_id,
        "symbol": "XAUUSD",
        "is_long": "True",
        "entry_time": (exit_time - pd.Timedelta(hours=1)).isoformat(),
        "exit_time": exit_time.isoformat(),
        "entry_price": 100.0,
        "exit_price": exit_price,
        "stop_price": 98.0,
        "profit": profit,
    }


tmp_dir = Path(tempfile.mkdtemp(prefix="themba_wf_demo_"))
trades_csv = tmp_dir / "trades.csv"
pd.DataFrame(
    [
        row("t0", BASE, 104.0, 10.0),
        row("t1", BASE + pd.Timedelta(days=1, hours=12), 99.0, -5.0),
        row("t2", BASE + pd.Timedelta(days=2, hours=12), 103.0, 10.0),
        row("t4", BASE + pd.Timedelta(days=4, hours=12), 105.0, 20.0),
        row("t8", BASE + pd.Timedelta(days=8, hours=12), 99.0, -5.0),
    ]
).to_csv(trades_csv, index=False)

In [ ]:
windows_df = run(
    trades_csv,
    train_days=3,
    test_days=2,
    step_days=2,
    summary_json=tmp_dir / "summary.json",
    repo_path=PROJECT_ROOT.parents[1],
)

print(
    windows_df[
        [
            "window_index",
            "train_n",
            "train_win_rate",
            "train_expectancy_r",
            "test_n",
            "test_win_rate",
            "test_expectancy_r",
        ]
    ]
)

# Hand-traced in tests/test_walk_forward.py::test_windows_hand_computed_with_purged_boundaries.
assert len(windows_df) == 3
assert windows_df.iloc[0]["train_n"] == 3  # t0, t1, t2 (t0 now anchors window 0)
assert abs(windows_df.iloc[0]["train_win_rate"] - (2.0 / 3.0)) < 1e-9
assert windows_df.iloc[0]["test_n"] == 1  # t4
assert abs(windows_df.iloc[0]["test_expectancy_r"] - 2.5) < 1e-9

## Real-data run: PENDING

No real trade history spanning multiple months exists yet -- see `walk_forward.py`'s own
documented scope (descriptive rolling stability, not parameter-selection walk-forward
optimization).